# SAT-GUARDIAN: Data Exploration
Explore synthetic satellite data, optical flow, and visualisation utilities.


In [ ]:
import sys
from pathlib import Path
ROOT = Path('..').resolve()
sys.path.insert(0, str(ROOT / 'src'))

import numpy as np
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['figure.dpi'] = 120
%matplotlib inline
print('Imports OK')


## 1. Generate Sample Data

In [ ]:
from data_loader import generate_sample_frames

data = generate_sample_frames(H=256, W=256, seed=42)
frame_t0, frame_t1 = data['frame_t0'], data['frame_t1']
wind_u, wind_v     = data['wind_u'],   data['wind_v']

print(f'T0  shape: {frame_t0.shape}, range: [{frame_t0.min():.3f}, {frame_t0.max():.3f}]')
print(f'T1  shape: {frame_t1.shape}, range: [{frame_t1.min():.3f}, {frame_t1.max():.3f}]')
print(f'Wind u  range: [{wind_u.min():.2f}, {wind_u.max():.2f}] m/s')
print(f'Wind v  range: [{wind_v.min():.2f}, {wind_v.max():.2f}] m/s')


In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(18, 4))
axes[0].imshow(frame_t0, cmap='gray'); axes[0].set_title('T₀ (Input)'); axes[0].axis('off')
axes[1].imshow(frame_t1, cmap='gray'); axes[1].set_title('T₁ (Input)'); axes[1].axis('off')
axes[2].imshow(wind_u, cmap='RdBu_r'); axes[2].set_title('ERA5 Wind U (m/s)'); axes[2].axis('off')
axes[3].imshow(wind_v, cmap='RdBu_r'); axes[3].set_title('ERA5 Wind V (m/s)'); axes[3].axis('off')
plt.tight_layout(); plt.show()


## 2. Physics-Constrained Optical Flow

In [ ]:
from optical_flow import compute_optical_flow, flow_to_rgb, flow_to_magnitude_angle

flow_fwd, opt_flow, wind_flow = compute_optical_flow(
    frame_t0, frame_t1, wind_u, wind_v,
    optical_weight=0.70, physics_weight=0.30
)

mag, ang = flow_to_magnitude_angle(flow_fwd)
print(f'Flow magnitude: mean={mag.mean():.3f}, max={mag.max():.3f} pixels')

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
axes[0].imshow(flow_to_rgb(opt_flow));  axes[0].set_title('Raw Optical Flow');       axes[0].axis('off')
axes[1].imshow(flow_to_rgb(wind_flow)); axes[1].set_title('ERA5 Wind Flow');          axes[1].axis('off')
axes[2].imshow(flow_to_rgb(flow_fwd));  axes[2].set_title('Physics-Constrained Flow');axes[2].axis('off')
plt.tight_layout(); plt.show()


## 3. Frame Generation (Cascade)

In [ ]:
from three_frame_generator import generate_all_frames

gen = generate_all_frames(frame_t0, frame_t1, wind_u, wind_v, strategy='flow')

fig, axes = plt.subplots(1, 5, figsize=(20, 4))
data_to_show = [
    (frame_t0,        'T₀ (Input)'),
    (gen['frame_025'], 'T₀.₂₅ (Gen)'),
    (gen['frame_050'], 'T₀.₅₀ (Gen)'),
    (gen['frame_075'], 'T₀.₇₅ (Gen)'),
    (frame_t1,         'T₁ (Input)'),
]
for ax, (frm, lbl) in zip(axes, data_to_show):
    ax.imshow(frm, cmap='gray', vmin=0, vmax=1)
    ax.set_title(lbl, fontsize=11, fontweight='bold')
    ax.axis('off')
plt.suptitle('SAT-GUARDIAN: Generated Intermediate Frames', fontsize=14, fontweight='bold')
plt.tight_layout(); plt.show()


## 4. Confidence Map

In [ ]:
from confidence_map import compute_confidence_map, confidence_stats

conf  = compute_confidence_map(gen['optical_flow_fwd'], gen['wind_flow_fwd'],
                               flow_bwd=gen['flow_bwd'], flow_fwd=gen['flow_fwd'])
stats = confidence_stats(conf)
print('Confidence stats:', {k: round(v,4) for k,v in stats.items()})

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].imshow(gen['frame_050'], cmap='gray'); axes[0].set_title('T₀.₅₀ Frame'); axes[0].axis('off')
im = axes[1].imshow(conf, cmap='RdYlGn', vmin=0, vmax=1)
plt.colorbar(im, ax=axes[1], label='Confidence')
axes[1].set_title('Pixel Confidence Map'); axes[1].axis('off')
plt.tight_layout(); plt.show()


## 5. Cloud Motion Score & Metrics

In [ ]:
from cloud_motion_score import score_report
from metrics import evaluate_frame

cm_report = score_report(gen['flow_fwd'], gen['wind_flow_fwd'])
print('Cloud Motion Score:', cm_report['overall_score'], '/100')
print('Interpretation    :', cm_report['interpretation'])

gt_050 = 0.5 * frame_t0 + 0.5 * frame_t1
m = evaluate_frame(gen['frame_050'], gt_050, 'T0.50')
print(f"\nMetrics: SSIM={m['ssim']:.4f} | PSNR={m['psnr']:.2f}dB | MSE={m['mse']:.5f} | FSIM={m['fsim']:.4f}")
